# Uncertainty Quantification (Phase 7)

For every dataset and every shared output target, this notebook computes out-of-fold Prediction Intervals (PI) for GPR plus the Top-3 models:

* **GPR (Kriging)** — analytical PI from the Gaussian Process posterior predictive std.
* **All other models (Top-3)** — percentile Bootstrap PI (resample the training fold, refit, collect the prediction distribution; `configs/config.yaml: uncertainty.bootstrap_iterations` resamples per fold).

Reported per (dataset, target, model): mean PI width and empirical coverage (fraction of true values inside the interval at the configured confidence level). These numbers also feed the Top-3 multi-criteria selection's uncertainty criterion.

Note: with a 95% confidence level and `bootstrap_iterations=200` refits per fold, the bootstrap models are the slowest part of this notebook — this can take a while to complete for all dataset/target/model combinations.

Setup: repo imports, pandas display options, load the Top-3 models and all datasets.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
SCRIPTS_DIR = REPO_ROOT / 'scripts'
for p in (REPO_ROOT, SCRIPTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

from _pipeline_common import read_top3_models
from src.config import get_base_seed, get_path, load_config
from src.data.loader import load_all_datasets
from src.evaluation import bootstrap_prediction_interval, gpr_prediction_interval, interval_metrics
from src.models.registry import build_model
from src.utils.io import save_table

cfg = load_config()
seed = get_base_seed()
n_splits = cfg['cross_validation']['n_splits']
models = sorted(set(read_top3_models()) | {'gpr'})
bundles = load_all_datasets(discrete=False)
print('Models evaluated (Top-3 union GPR):', models)
print('Confidence level:', cfg['uncertainty']['confidence_level'])
print('Bootstrap iterations per fold:', cfg['uncertainty']['bootstrap_iterations'])

Models evaluated (Top-3 union GPR): ['gpr', 'lightgbm', 'xgboost']
Confidence level: 0.95
Bootstrap iterations per fold: 200


## Prediction intervals on Dataset_0136

For every shared target and every model: 5-fold CV, GPR gets the analytical PI, everything else gets the bootstrap PI; predictions across folds are concatenated before computing PI width and coverage.

In [2]:
rows_0136 = []
bundle = bundles['Dataset_0136']
X = bundle.X.to_numpy()
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target].to_numpy()
    for model_name in models:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        y_true_all, lo_all, hi_all = [], [], []
        for tr, te in kf.split(X):
            pipe = build_model(model_name, seed=seed)
            pipe.fit(X[tr], y[tr])
            if model_name == 'gpr':
                _, lo, hi = gpr_prediction_interval(pipe, X[te])
            else:
                _, lo, hi = bootstrap_prediction_interval(
                    build_model(model_name, seed=seed), X[tr], y[tr], X[te], seed=seed
                )
            y_true_all.append(y[te]); lo_all.append(lo); hi_all.append(hi)
        metrics = interval_metrics(
            np.concatenate(y_true_all), np.concatenate(lo_all), np.concatenate(hi_all)
        )
        rows_0136.append({'dataset': 'Dataset_0136', 'target': target, 'model': model_name, **metrics})
        print(f'[uncertainty] Dataset_0136 / {target} / {model_name}: {metrics}')

uncertainty_0136 = pd.DataFrame(rows_0136)
uncertainty_0136

[uncertainty] Dataset_0136 / Si Particle Size (um) / gpr: {'pi_mean_width': 0.8531588351786767, 'pi_coverage': 0.8611111111111112}


[uncertainty] Dataset_0136 / Si Particle Size (um) / lightgbm: {'pi_mean_width': 0.6475931748007535, 'pi_coverage': 0.8888888888888888}


[uncertainty] Dataset_0136 / Si Particle Size (um) / xgboost: {'pi_mean_width': 0.465142002536191, 'pi_coverage': 0.8611111111111112}


[uncertainty] Dataset_0136 / Hardness (HV) / gpr: {'pi_mean_width': 5.801828728768703, 'pi_coverage': 0.9722222222222222}


[uncertainty] Dataset_0136 / Hardness (HV) / lightgbm: {'pi_mean_width': 4.759705619362713, 'pi_coverage': 0.8611111111111112}


...

C:\Users\mohammadhosein\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[uncertainty] Dataset_0136 / Elongation (%) / gpr: {'pi_mean_width': 6.82978615606823, 'pi_coverage': 0.9444444444444444}


[uncertainty] Dataset_0136 / Elongation (%) / lightgbm: {'pi_mean_width': 4.327413558662856, 'pi_coverage': 0.8888888888888888}


[uncertainty] Dataset_0136 / Elongation (%) / xgboost: {'pi_mean_width': 4.497309025790957, 'pi_coverage': 0.75}


,dataset,target,model,pi_mean_width,pi_coverage
0,Dataset_0136,Si Particle Size (um),gpr,0.853159,0.861111
1,Dataset_0136,Si Particle Size (um),lightgbm,0.647593,0.888889
2,Dataset_0136,Si Particle Size (um),xgboost,0.465142,0.861111
3,Dataset_0136,Hardness (HV),gpr,5.801829,0.972222
4,Dataset_0136,Hardness (HV),lightgbm,4.759706,0.861111
5,Dataset_0136,Hardness (HV),xgboost,3.584789,0.805556
6,Dataset_0136,wear rate,gpr,1.079015,0.972222
7,Dataset_0136,wear rate,lightgbm,0.771643,0.861111
8,Dataset_0136,wear rate,xgboost,0.579710,0.861111
9,Dataset_0136,Ultimate Compression Strength (MPa),gpr,16.363094,0.972222


## Prediction intervals on Dataset_0172

For every shared target and every model: 5-fold CV, GPR gets the analytical PI, everything else gets the bootstrap PI; predictions across folds are concatenated before computing PI width and coverage.

In [3]:
rows_0172 = []
bundle = bundles['Dataset_0172']
X = bundle.X.to_numpy()
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target].to_numpy()
    for model_name in models:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        y_true_all, lo_all, hi_all = [], [], []
        for tr, te in kf.split(X):
            pipe = build_model(model_name, seed=seed)
            pipe.fit(X[tr], y[tr])
            if model_name == 'gpr':
                _, lo, hi = gpr_prediction_interval(pipe, X[te])
            else:
                _, lo, hi = bootstrap_prediction_interval(
                    build_model(model_name, seed=seed), X[tr], y[tr], X[te], seed=seed
                )
            y_true_all.append(y[te]); lo_all.append(lo); hi_all.append(hi)
        metrics = interval_metrics(
            np.concatenate(y_true_all), np.concatenate(lo_all), np.concatenate(hi_all)
        )
        rows_0172.append({'dataset': 'Dataset_0172', 'target': target, 'model': model_name, **metrics})
        print(f'[uncertainty] Dataset_0172 / {target} / {model_name}: {metrics}')

uncertainty_0172 = pd.DataFrame(rows_0172)
uncertainty_0172

[uncertainty] Dataset_0172 / Si Particle Size (um) / gpr: {'pi_mean_width': 0.6240813926820503, 'pi_coverage': 0.8888888888888888}


[uncertainty] Dataset_0172 / Si Particle Size (um) / lightgbm: {'pi_mean_width': 0.4849490690106017, 'pi_coverage': 0.9305555555555556}


[uncertainty] Dataset_0172 / Si Particle Size (um) / xgboost: {'pi_mean_width': 0.3845218759547505, 'pi_coverage': 0.8888888888888888}


[uncertainty] Dataset_0172 / Hardness (HV) / gpr: {'pi_mean_width': 6.611251987851937, 'pi_coverage': 0.9444444444444444}


[uncertainty] Dataset_0172 / Hardness (HV) / lightgbm: {'pi_mean_width': 4.026378544951773, 'pi_coverage': 0.8333333333333334}


...

C:\Users\mohammadhosein\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[uncertainty] Dataset_0172 / Elongation (%) / gpr: {'pi_mean_width': 7.661939269095959, 'pi_coverage': 0.9027777777777778}


[uncertainty] Dataset_0172 / Elongation (%) / lightgbm: {'pi_mean_width': 3.871361325501153, 'pi_coverage': 0.7638888888888888}


[uncertainty] Dataset_0172 / Elongation (%) / xgboost: {'pi_mean_width': 3.6546037683884296, 'pi_coverage': 0.7361111111111112}


,dataset,target,model,pi_mean_width,pi_coverage
0,Dataset_0172,Si Particle Size (um),gpr,0.624081,0.888889
1,Dataset_0172,Si Particle Size (um),lightgbm,0.484949,0.930556
2,Dataset_0172,Si Particle Size (um),xgboost,0.384522,0.888889
3,Dataset_0172,Hardness (HV),gpr,6.611252,0.944444
4,Dataset_0172,Hardness (HV),lightgbm,4.026379,0.833333
5,Dataset_0172,Hardness (HV),xgboost,3.263141,0.888889
6,Dataset_0172,wear rate,gpr,0.889266,0.972222
7,Dataset_0172,wear rate,lightgbm,0.696921,0.902778
8,Dataset_0172,wear rate,xgboost,0.491805,0.847222
9,Dataset_0172,Ultimate Compression Strength (MPa),gpr,20.671950,0.930556


## Prediction intervals on Dataset_3772

For every shared target and every model: 5-fold CV, GPR gets the analytical PI, everything else gets the bootstrap PI; predictions across folds are concatenated before computing PI width and coverage.

In [4]:
rows_3772 = []
bundle = bundles['Dataset_3772']
X = bundle.X.to_numpy()
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target].to_numpy()
    for model_name in models:
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        y_true_all, lo_all, hi_all = [], [], []
        for tr, te in kf.split(X):
            pipe = build_model(model_name, seed=seed)
            pipe.fit(X[tr], y[tr])
            if model_name == 'gpr':
                _, lo, hi = gpr_prediction_interval(pipe, X[te])
            else:
                _, lo, hi = bootstrap_prediction_interval(
                    build_model(model_name, seed=seed), X[tr], y[tr], X[te], seed=seed
                )
            y_true_all.append(y[te]); lo_all.append(lo); hi_all.append(hi)
        metrics = interval_metrics(
            np.concatenate(y_true_all), np.concatenate(lo_all), np.concatenate(hi_all)
        )
        rows_3772.append({'dataset': 'Dataset_3772', 'target': target, 'model': model_name, **metrics})
        print(f'[uncertainty] Dataset_3772 / {target} / {model_name}: {metrics}')

uncertainty_3772 = pd.DataFrame(rows_3772)
uncertainty_3772

C:\Users\mohammadhosein\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[uncertainty] Dataset_3772 / Si Particle Size (um) / gpr: {'pi_mean_width': 0.22651388694158203, 'pi_coverage': 0.8888888888888888}


[uncertainty] Dataset_3772 / Si Particle Size (um) / lightgbm: {'pi_mean_width': 0.1514013084110462, 'pi_coverage': 0.7222222222222222}


[uncertainty] Dataset_3772 / Si Particle Size (um) / xgboost: {'pi_mean_width': 0.12874450516990488, 'pi_coverage': 0.6944444444444444}


[uncertainty] Dataset_3772 / Hardness (HV) / gpr: {'pi_mean_width': 7.278501471673181, 'pi_coverage': 0.8611111111111112}


...

C:\Users\mohammadhosein\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[uncertainty] Dataset_3772 / Elongation (%) / gpr: {'pi_mean_width': 9.222706717989839, 'pi_coverage': 0.8611111111111112}


[uncertainty] Dataset_3772 / Elongation (%) / lightgbm: {'pi_mean_width': 5.446962504535235, 'pi_coverage': 0.8055555555555556}


[uncertainty] Dataset_3772 / Elongation (%) / xgboost: {'pi_mean_width': 5.404791428645453, 'pi_coverage': 0.8055555555555556}


,dataset,target,model,pi_mean_width,pi_coverage
0,Dataset_3772,Si Particle Size (um),gpr,0.226514,0.888889
1,Dataset_3772,Si Particle Size (um),lightgbm,0.151401,0.722222
2,Dataset_3772,Si Particle Size (um),xgboost,0.128745,0.694444
3,Dataset_3772,Hardness (HV),gpr,7.278501,0.861111
4,Dataset_3772,Hardness (HV),lightgbm,5.425558,0.750000
5,Dataset_3772,Hardness (HV),xgboost,4.175392,0.861111
6,Dataset_3772,wear rate,gpr,0.747988,0.916667
7,Dataset_3772,wear rate,lightgbm,0.605564,0.861111
8,Dataset_3772,wear rate,xgboost,0.459820,0.861111
9,Dataset_3772,Ultimate Compression Strength (MPa),gpr,25.341343,0.944444


## Final Summary — Complete Prediction Interval Results (All Datasets)

PI width and coverage for every (dataset, target, model) combination, saved to `results/uncertainty/prediction_intervals.csv`.

In [5]:
uncertainty_all = pd.concat([uncertainty_0136, uncertainty_0172, uncertainty_3772], ignore_index=True)
save_table(uncertainty_all, get_path('results_dir') / 'uncertainty' / 'prediction_intervals.csv')

print('=' * 90)
print('PREDICTION INTERVAL RESULTS — PI width and empirical coverage (all datasets)')
print('=' * 90)
display(uncertainty_all)

print()
print('Mean PI width and coverage by model (averaged over datasets/targets):')
display(uncertainty_all.groupby('model')[['pi_mean_width', 'pi_coverage']].mean())

print()
print('Uncertainty results saved to:', (get_path('results_dir') / 'uncertainty').resolve())

PREDICTION INTERVAL RESULTS — PI width and empirical coverage (all datasets)


,dataset,target,model,pi_mean_width,pi_coverage
0,Dataset_0136,Si Particle Size (um),gpr,0.853159,0.861111
1,Dataset_0136,Si Particle Size (um),lightgbm,0.647593,0.888889
2,Dataset_0136,Si Particle Size (um),xgboost,0.465142,0.861111
3,Dataset_0136,Hardness (HV),gpr,5.801829,0.972222
4,Dataset_0136,Hardness (HV),lightgbm,4.759706,0.861111
5,Dataset_0136,Hardness (HV),xgboost,3.584789,0.805556
6,Dataset_0136,wear rate,gpr,1.079015,0.972222
7,Dataset_0136,wear rate,lightgbm,0.771643,0.861111
8,Dataset_0136,wear rate,xgboost,0.579710,0.861111
9,Dataset_0136,Ultimate Compression Strength (MPa),gpr,16.363094,0.972222



Mean PI width and coverage by model (averaged over datasets/targets):


,pi_mean_width,pi_coverage
model,,
gpr,7.346828,0.922222
lightgbm,5.673052,0.845370
xgboost,4.905583,0.834259



Uncertainty results saved to: C:\Users\mohammadhosein\Desktop\DeepLearning-miniProject\results\uncertainty
